# 13 - Chonkie Chunking

This notebook explores Chonkie text chunking inside JupyterHub and compares it with the canonical Atlas Backend runtime endpoint at `/api/chunk`. Use the local cells for quick experiments, then use the Backend endpoint when a workflow needs stable Atlas-wide defaults and response metadata.

In [ ]:
import json
import os

import httpx
from chonkie import RecursiveChunker, TokenChunker

SAMPLE_TEXT = """
Atlas routes retrieval workloads through a shared Backend API so notebooks,
workflow automations, and services can reuse the same chunking defaults. Token
chunking is useful when a caller needs predictable model-window boundaries.
Recursive chunking tries to preserve natural text boundaries such as paragraphs
and sentences. Semantic chunking can group text by meaning when an embedding
model is available.
""".strip()

def show_chunks(name, chunks):
    rows = []
    for index, chunk in enumerate(chunks):
        rows.append({
            "strategy": name,
            "index": index,
            "start": getattr(chunk, "start_index", None),
            "end": getattr(chunk, "end_index", None),
            "tokens": getattr(chunk, "token_count", None),
            "text": getattr(chunk, "text", str(chunk)),
        })
    print(json.dumps(rows, indent=2))

## 1. Local Chonkie Experiments

These cells use the package installed in the JupyterHub image. They are useful for exploring strategy behavior before moving a workflow to the Backend API.

In [ ]:
token_chunker = TokenChunker(tokenizer="gpt2", chunk_size=48, chunk_overlap=8)
recursive_chunker = RecursiveChunker(
    tokenizer="gpt2",
    chunk_size=120,
    min_characters_per_chunk=24,
)

show_chunks("token", token_chunker(SAMPLE_TEXT, show_progress_bar=False))
show_chunks("recursive", recursive_chunker(SAMPLE_TEXT, show_progress=False))

In [ ]:
if os.getenv("RUN_SEMANTIC_CHONKIE") == "1":
    from chonkie import SemanticChunker

    semantic_chunker = SemanticChunker(
        embedding_model=os.getenv("CHONKIE_SEMANTIC_EMBEDDING_MODEL", "minishlab/potion-base-32M"),
        threshold=0.8,
        chunk_size=160,
        min_characters_per_sentence=24,
    )
    show_chunks("semantic", semantic_chunker(SAMPLE_TEXT))
else:
    print("Set RUN_SEMANTIC_CHONKIE=1 to run semantic chunking. The first run may download an embedding model.")

## 2. Backend Runtime Endpoint

Use `POST /api/chunk` when a notebook, n8n workflow, or service needs the shared Atlas chunking contract. The endpoint returns ordered chunks, stable character offsets, token counts where available, and strategy metadata.

In [ ]:
backend_url = os.getenv("BACKEND_API_URL", "http://backend:8000").rstrip("/")
backend_token = os.getenv("BACKEND_NOTEBOOK_API_TOKEN")
backend_headers = {"Authorization": f"Bearer {backend_token}"} if backend_token else {}
payload = {
    "text": SAMPLE_TEXT,
    "strategy": "token",
    "chunk_size": 48,
    "overlap": 8,
    "tokenizer": "gpt2",
}

try:
    response = httpx.post(f"{backend_url}/api/chunk", headers=backend_headers, json=payload, timeout=20.0)
    response.raise_for_status()
    result = response.json()
    print(json.dumps(result, indent=2))
except httpx.HTTPError as exc:
    print(f"Backend chunking request failed: {exc}")
    print("Confirm BACKEND_SOURCE=container and that the backend service is healthy.")

## 3. Workflow Notes

- Keep experimentation in JupyterHub when tuning strategies on representative text.
- Promote production ingestion workflows to the Backend `/api/chunk` endpoint.
- n8n and future document-ingestion services should call the Backend endpoint rather than adding their own Chonkie dependency.
- Use semantic chunking deliberately because it may require an embedding model download or a configured local cache.